---
## 4. Preprocesamiento de datos <a id='4'></a>

Los modelos de ML solo entienden números. Convertimos variables categóricas y escalamos las numéricas.

| Técnica | Cuándo usarla |
|---|---|
| **Label Encoding** | Variables binarias o con orden natural (low < medium < high) |
| **One-Hot Encoding** | Variables nominales sin orden (Instagram, TikTok, Both, Other) |
| **MinMaxScaler** | Escala a [0,1] — útil para KNN y Redes Neuronales |
| **StandardScaler** | Media=0, std=1 — mejor para Regresión Logística y SVM |


In [ ]:
# ── 4.1 Codificación de variables categóricas ─────────────────────────────
# Guardamos una copia limpia SIN escalar ni codificar (para comparar luego)
df_clean = df.copy()

# ── Variable OBJETIVO: Label Encoding ───────────────────────────────────
le_target = LabelEncoder()
df_clean['depression_risk_enc'] = le_target.fit_transform(df['depression_risk'])
print("Codificación de depression_risk:")
for orig, enc in zip(le_target.classes_, range(len(le_target.classes_))):
    print(f"  '{orig}' → {enc}")

# ── GENDER: Label Encoding (solo 2 valores → equivalente a One-Hot) ────────
df_clean['gender_enc'] = LabelEncoder().fit_transform(df['gender'])
print(f"\ngender: {dict(zip(df['gender'].unique(), LabelEncoder().fit_transform(df['gender'].unique())))}")

# ── SOCIAL_INTERACTION_LEVEL: Label Encoding ORDINAL ──────────────────────
# Hay un orden claro: low < medium < high → preservamos ese orden con mapeo manual
sil_map = {'low': 0, 'medium': 1, 'high': 2}
df_clean['social_interaction_enc'] = df['social_interaction_level'].map(sil_map)
print(f"\nsocial_interaction_level (ordinal): {sil_map}")

# ── PLATFORM_USAGE: One-Hot Encoding ──────────────────────────────────────
# Variable nominal: no hay orden entre Instagram, TikTok, Both, Other
# drop_first=True elimina una columna para evitar multicolinealidad perfecta
dummies = pd.get_dummies(df['platform_usage'], prefix='platform', drop_first=True)
df_clean = pd.concat([df_clean, dummies], axis=1)
print(f"\nplatform_usage → columnas OHE: {list(dummies.columns)}")

In [ ]:
# ── 4.2 Definir X e y ────────────────────────────────────────────────────
FEAT = ['age', 'gender_enc', 'daily_social_media_hours', 'sleep_hours',
        'screen_time_before_sleep', 'academic_performance', 'physical_activity',
        'social_interaction_enc', 'stress_level', 'anxiety_level'
        ] + list(dummies.columns)

# X_raw: datos codificados SIN escalar (útil para árboles de decisión)
X_raw = df_clean[FEAT]
y     = df_clean['depression_risk_enc']

print(f"X_raw shape: {X_raw.shape}")
print(f"y shape:     {y.shape}")
print(f"\nClases: {list(zip(range(3), le_target.classes_))}")

# División train / test estratificada 80/20
# stratify=y garantiza misma proporción de clases en train y test
X_train, X_test, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=42, stratify=y)

print(f"\nTrain: {X_train.shape[0]} muestras | Test: {X_test.shape[0]} muestras")

In [ ]:
# ── 4.3 Normalización y Estandarización ──────────────────────────────────

# MinMaxScaler → rango [0, 1]
# Fórmula: (x - min) / (max - min)
# Pros: preserva distribución original. Contra: sensible a outliers extremos.
scaler_mm  = MinMaxScaler()
X_minmax   = pd.DataFrame(scaler_mm.fit_transform(X_raw), columns=FEAT)

# StandardScaler → media=0, std=1
# Fórmula: (x - media) / desviacion_estandar
# Pros: robusto, funciona bien con modelos lineales y SVM
scaler_std = StandardScaler()
X_std      = pd.DataFrame(scaler_std.fit_transform(X_raw), columns=FEAT)

# Dividir versiones escaladas también
X_train_mm,  X_test_mm,  _, _ = train_test_split(X_minmax, y, test_size=0.2, random_state=42, stratify=y)
X_train_std, X_test_std, _, _ = train_test_split(X_std,    y, test_size=0.2, random_state=42, stratify=y)

# Comparación visual: misma columna antes y después
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
col_demo  = 'stress_level'
for ax, data, title in zip(axes,
    [X_raw, X_minmax, X_std],
    ['Original', 'MinMaxScaler [0,1]', 'StandardScaler (μ=0, σ=1)']):
    ax.hist(data[col_demo], bins=25, color='#5C6BC0', alpha=0.8, edgecolor='white')
    ax.set_title(f"{title}\n'{col_demo}'", fontsize=11, fontweight='bold')
    ax.set_xlabel('Valor')
    ax.set_ylabel('Frecuencia')
plt.suptitle('Efecto del escalado sobre stress_level', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig6_scaling_comparison.png', dpi=130, bbox_inches='tight')
plt.show()

print("Nota: La FORMA de la distribución no cambia, solo la escala del eje X.")
print("Esto es clave: el escalado no crea información nueva, solo re-expresa los valores.")